# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('/content/hotels.csv')

df = df.drop(columns=['company', 'agent', 'reservation_status_date'])

df['country'] = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0)

le = LabelEncoder()
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['is_canceled', 'reservation_status'])
y = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Dataset shape: {df.shape}")

Dataset shape: (9404, 29)


### ✍️ Your Response: 🔧
1.There are 9404 rows and 29 columns

2. Numerical like lead time, adr, adult numbers and children. Categorical: hotel, meal, market_segment

3. Dropped company, agent, reservation status date because too many missing values. Handled missing data by lebeling unknown countries as unknown and assuming 0 for missing children counts. Used labelencoder to swap text categores into numbers so models can process them.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [3]:
# Add code here 🔧
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

y_pred = nb_model.predict(X_test)

print("--- Naïve Bayes Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Naïve Bayes Classification Report ---
              precision    recall  f1-score   support

           0       0.96      0.70      0.81      2111
           1       0.51      0.91      0.65       711

    accuracy                           0.76      2822
   macro avg       0.73      0.81      0.73      2822
weighted avg       0.85      0.76      0.77      2822

--- Confusion Matrix ---
[[1484  627]
 [  64  647]]


### ✍️ Your Response: 🔧
1. Performance was quite accurate at 76%. The best metric was recall because it's better to flag a guest who might cancel and be ready for it than to be totally blindsided

2. Since naïve bayes is fast the hotel could use it to instantly flag high-risk bookings. Because this model is so good at catching potential cancellations management can use these insights to overbook certain dates.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [4]:
# Add code here 🔧
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(kernel='linear')
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)

print("--- SVM Classification Report ---")
print(classification_report(y_test, y_pred_svm))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_svm))

--- SVM Classification Report ---
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      2111
           1       0.88      0.56      0.68       711

    accuracy                           0.87      2822
   macro avg       0.87      0.77      0.80      2822
weighted avg       0.87      0.87      0.86      2822

--- Confusion Matrix ---
[[2056   55]
 [ 313  398]]


### ✍️ Your Response: 🔧
1. The performance improved a lot all is good at confirming guest, who won't cancel, it only catches about 56% of the actual cancellations. The best metric is precision at around 88%.

2. Because SVM is more precise, it's better for decisions that cost the hotel money. You don't want to annoy a loyal guest with retention offers unless you're sure they might leave. Since SVM handles complex patterns, better than simpler models, it's great for long-term revenue forecasting.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [5]:
# Add code here 🔧
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=1000, random_state=42)

nn_model.fit(X_train_scaled, y_train)

y_pred_nn = nn_model.predict(X_test_scaled)

print("--- Neural Network Classification Report ---")
print(classification_report(y_test, y_pred_nn))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_nn))

--- Neural Network Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      2111
           1       0.84      0.75      0.79       711

    accuracy                           0.90      2822
   macro avg       0.88      0.85      0.87      2822
weighted avg       0.90      0.90      0.90      2822

--- Confusion Matrix ---
[[2009  102]
 [ 175  536]]


### ✍️ Your Response: 🔧
1. This is the strongest model with around 90% accuracy. It caught 75% of the cancellations while keeping false alarm is low. It's definitely complex while the other models are more straightforward.

2. A business might be hesitant at first because he can't easily see why the model made a choice. Most managers are OK with this if it delivers higher revenue.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [6]:
# Add code here 🔧

print(f"Naïve Bayes Accuracy: 0.76")
print(f"SVM Accuracy: 0.87")
print(f"Neural Network Accuracy: 0.90")

models = ['Naïve Bayes', 'SVM', 'Neural Network']
accuracy = [0.76, 0.87, 0.90]
best_model = models[accuracy.index(max(accuracy))]

print(f"\nWinner: {best_model} is the most accurate.")

Naïve Bayes Accuracy: 0.76
SVM Accuracy: 0.87
Neural Network Accuracy: 0.90

Winner: Neural Network is the most accurate.


### ✍️ Your Response: 🔧
1. Best accuracy is the neural network, the best training time is naïve Bayes, the easiest to use is naïve bayes.

2. I'd recommend the neuron network. Even though it's complex, it has higher accuracy and getting better forecasting the high revenue.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend implanting the neural network for predicting booking cancellations. It scored the highest accuracy at 90% and provided the best balance between precision and recall. It reduced a lot of false alarms and caught most cancellations. This solves the business problem of revenue loss due to unpredictable occupancy. Well, you can't see what happens and it's hard to interpret the accuracy boost justifies if we added more data like customer loyalty or historical weather patterns we can make the predictions even better.

2. This relates to my goal of learning more about Python based analytics because I'm learning to successfully build and compare three types of machine, learning models I can now handle and end.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [7]:
!jupyter nbconvert --to html "assignment_12_erosolowski.ipynb"

[NbConvertApp] Converting notebook assignment_12_erosolowski.ipynb to html
[NbConvertApp] Writing 317722 bytes to assignment_12_erosolowski.html
